# Stage 4 — the single evaluation on DS2

**Runtime → Change runtime type → T4 GPU → Save. Then Runtime → Run all.**

About **45 minutes**, most of it unattended. Cell 5 is the one that matters.

---

### Read this before running anything

DS2 is twenty-two patients that have been sealed since the start of this
project. Not one setting, model choice or threshold anywhere in the work was
picked by looking at them. That is the only reason the number this notebook
produces means anything.

**It only works once.** A test score you are allowed to re-run until it
improves is not a test score, it is a tuned number wearing a test score's
clothes. The script enforces this rather than trusting it: the first run
writes a receipt, and every later run is refused unless you give a written
reason, which is appended to that receipt.

So: if something looks wrong, **stop and report it rather than re-running**.
A failed run that never reached the evaluation is fine and costs nothing. A
second evaluation is a thing that has to be explained in the paper.

### What this notebook does

1. Clones the public repository.
2. Downloads MIT-BIH from PhysioNet and rebuilds the dataset — you are not
   sent any data, you regenerate it, and 21 acceptance checks have to pass.
3. Trains four locked configurations on DS1 and predicts DS2 **once**.
4. Reports AAMI sensitivity and positive predictivity per class, macro-F1,
   and the confusion matrix.

The configurations are fixed in the script and there is deliberately no flag
to change them.

## 1. Confirm a GPU was allocated

In [1]:
import torch

assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.')

print('GPU   :', torch.cuda.get_device_name(0))
print('torch :', torch.__version__)

GPU   : Tesla T4
torch : 2.11.0+cu130


## 2. Get the code

Public repository — nothing to request and nothing to upload.

In [9]:
%cd /content
!rm -rf ecg-paper
!git clone --depth 1 https://github.com/Rishabhhcodes/ecg-paper.git
%cd /content/ecg-paper

!pip install -q wfdb xgboost imbalanced-learn

import subprocess
print('\ncommit :', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                                    capture_output=True, text=True).stdout.strip())

/content
Cloning into 'ecg-paper'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 37 (delta 1), reused 31 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 99.90 KiB | 519.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/ecg-paper

commit : 1d69436


In [10]:
import glob, os, shutil, zipfile
import numpy as np

EXPECT = {'ds1.npz': 50977, 'ds2.npz': 49668}
os.makedirs('artifacts', exist_ok=True)

# 1. Clean up any loose .npy files from previous extraction attempts
for stray in ('X.npy', 'rr.npy', 'y.npy', 'pid.npy'):
    stray_path = os.path.join('/content/ecg-paper', stray)
    if os.path.exists(stray_path):
        os.remove(stray_path)

from google.colab import files
uploaded_files = files.upload()                    # Select and upload BOTH your zip or npz files

# 2. Extract zip archives
for name in glob.glob('*.zip'):
    with zipfile.ZipFile(name) as z:
        z.extractall('.')

    # If the zip extracted raw .npy files, bundle them into a clean .npz
    extracted_npy = ['X.npy', 'rr.npy', 'y.npy', 'pid.npy']
    if all(os.path.exists(f) for f in extracted_npy):
        target_npz_name = name.replace('.zip', '')  # e.g., ds1.npz or ds2.npz
        data_dict = {f.replace('.npy', ''): np.load(f) for f in extracted_npy}
        np.savez(target_npz_name, **data_dict)
        # Clean up loose files
        for f in extracted_npy:
            os.remove(f)
    os.remove(name)

# 3. Move all actual .npz files into 'artifacts'
for name in glob.glob('*.npz'):
    shutil.move(name, os.path.join('artifacts', os.path.basename(name)))

print()
for f, n in EXPECT.items():
    p = os.path.join('artifacts', f)
    assert os.path.exists(p), (
        f'{p} missing. Re-run this cell and upload BOTH zip files.')
    d = np.load(p)
    assert sorted(d.files) == ['X', 'pid', 'rr', 'y'], f'{f}: {sorted(d.files)}'
    assert len(d['y']) == n, f'{f} has {len(d["y"]):,} beats, expected {n:,}'
    print(f'  {f}  {os.path.getsize(p)/1e6:5.1f} MB  {len(d["y"]):,} beats  OK')

print('\nBoth files in place. Run the verify cell next.')

Saving ds1.npz.zip to ds1.npz.zip
Saving ds2.npz.zip to ds2.npz.zip

  ds1.npz   45.7 MB  50,977 beats  OK
  ds2.npz   44.5 MB  49,668 beats  OK

Both files in place. Run the verify cell next.


## 3. Rebuild the dataset

Downloads about 90 MB from PhysioNet and rebuilds `ds1.npz` and `ds2.npz`
from it. Roughly 5 minutes.

You are rebuilding rather than being sent the files on purpose: the build is
deterministic, and the 21 checks in the next cell are what prove the data you
have is the data everyone else has.

## 4. Verify — stop here if anything fails

All 21 checks must pass. The expected counts are printed so you can read them
yourself rather than trusting a tick.

In [11]:
!python src/verify_dataset.py


[PROTOCOL] — failures here invalidate the whole study
  PASS  DS1 and DS2 share no patient
  PASS  No paced record present
  PASS  DS1 record set matches de Chazal 2004
  PASS  DS2 record set matches de Chazal 2004
  PASS  44 records total

[SHAPE]
  PASS  DS1 arrays aligned
  PASS  DS1 window length == 216
  PASS  DS1 has 4 RR features
  PASS  DS1 labels in 0..4
  PASS  DS1 no NaN / Inf
  PASS  DS2 arrays aligned
  PASS  DS2 window length == 216
  PASS  DS2 has 4 RR features
  PASS  DS2 labels in 0..4
  PASS  DS2 no NaN / Inf

[NORMALISATION]
  PASS  DS1 amplitude z-scored (|mean| < 1)
  PASS  DS2 amplitude z-scored (|mean| < 1)

[RR FEATURES] — the Class S diagnostic
  PASS  DS1 S beats are premature vs N (local ratio)
        DS1  N local_ratio=1.029  S local_ratio=0.797  separation=+0.231
  PASS  DS2 S beats are premature vs N (local ratio)
        DS2  N local_ratio=1.036  S local_ratio=0.718  separation=+0.318

[SCALE] — informational, compare against literature
  PASS  DS1 tota

In [13]:
import numpy as np
import sys
sys.path.insert(0, 'src')
import config as C

d1 = np.load('artifacts/ds1.npz')
d2 = np.load('artifacts/ds2.npz')

print('DS1', f"{len(d1['y']):,} beats /", len(np.unique(d1['pid'])), 'patients')
print('DS2', f"{len(d2['y']):,} beats /", len(np.unique(d2['pid'])), 'patients')
for name, d in (('DS1', d1), ('DS2', d2)):
    print(f'  {name}:', '  '.join(
        f'{c} {int((d["y"] == C.CLASS_TO_INT[c]).sum()):,}'
        for c in C.CLASS_NAMES))

assert len(d1['y']) == 50977, 'DS1 is the wrong size — do not continue'
assert len(d2['y']) == 49668, 'DS2 is the wrong size — do not continue'
assert not (set(np.unique(d1['pid'])) & set(np.unique(d2['pid']))), \
    'a patient appears in both sets — stop and report this'
print('\nOK — DS1 and DS2 share no patient.')

DS1 50,977 beats / 22 patients
DS2 49,668 beats / 22 patients
  DS1: N 45,824  S 943  V 3,788  F 414  Q 8
  DS2: N 44,218  S 1,836  V 3,219  F 388  Q 7

OK — DS1 and DS2 share no patient.


## 5. Optional — the Stage 2 predictions

Only needed for one secondary result: whether a temperature fitted on DS1
still calibrates predictions on unseen patients. **Skip this cell if Rishabh
has not sent you `stage4_preds.zip`** — everything else runs without it and
the script says so in its output.

These files cannot contain anything from DS2; they were written long before
this stage existed.

In [ ]:
import os
import zipfile

from google.colab import files

print('Select stage4_preds.zip, or run the next cell to skip.')
up = files.upload()

for name in up:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('.')
        os.remove(name)

import glob
n = len(glob.glob('artifacts/preds*/cnn_*_rr_f*.npz'))
print(f'\n{n} prediction files in place', '(expected 50)' if n else '')

## 6. The evaluation

**This is the cell that opens DS2.** About 25 minutes.

Four configurations: the CNN with class weighting and with focal loss — five
seeds each, because the two could not be separated in Stage 2 and picking one
would be an arbitrary choice presented as a result — plus XGBoost and logistic
regression once each.

One DS1 fold is held out for early stopping so the stopping epoch is never
chosen using DS2. Which fold rotates with the seed, so across five seeds all
five are used.

In [25]:
# -u is load-bearing: without it Python block-buffers stdout when it is not
# writing to a terminal, and this cell shows nothing at all for 25 minutes.
!python -u src/stage4_final.py --confirm

STAGE 4 — THE SINGLE EVALUATION ON DS2
DS1  50,977 beats / 22 patients
DS2  49,668 beats / 22 patients
DS2 class counts: N 44,218  S 1,836  V 3,219  F 388  Q 7
commit 1d69436ec1

--------------------------------------------------------------------------
cnn_class_weight_rr   (5 run(s))
--------------------------------------------------------------------------
  seed 0  macro-F1 0.6068   S Se 0.819/+P 0.414  V Se 0.885/+P 0.705   [55s]
  seed 1  macro-F1 0.6151   S Se 0.938/+P 0.373  V Se 0.912/+P 0.935   [43s]
  seed 2  macro-F1 0.6283   S Se 0.899/+P 0.330  V Se 0.919/+P 0.870   [41s]
  seed 3  macro-F1 0.6108   S Se 0.915/+P 0.318  V Se 0.848/+P 0.964   [43s]
  seed 4  macro-F1 0.6295   S Se 0.814/+P 0.497  V Se 0.934/+P 0.930   [60s]
  mean macro-F1 0.6181 ± 0.0113

--------------------------------------------------------------------------
cnn_focal_rr   (5 run(s))
--------------------------------------------------------------------------
  seed 0  macro-F1 0.5445   S Se 0.905/+P 0.

In [22]:
%cd /content/ecg-paper
!ls -la artifacts
!ls -l src/stage4_final.py

/content/ecg-paper
total 88080
drwxr-xr-x 2 root root     4096 Oct 22 17:57 .
drwxr-xr-x 7 root root     4096 Oct 22 17:57 ..
-rw-r--r-- 1 root root 45676356 Oct 22 17:57 ds1.npz
-rw-r--r-- 1 root root 44503492 Oct 22 17:57 ds2.npz
-rw-r--r-- 1 root root 17975 Oct 22 17:56 src/stage4_final.py


In [23]:
%cd /content/ecg-paper
!find artifacts -maxdepth 3 -type f

/content/ecg-paper
artifacts/ds1.npz
artifacts/ds2.npz


## 7. Package the results

In [26]:
import os
import zipfile

from google.colab import files

NEED = ['stage4_results.json', 'RECEIPT.json']
missing = [f for f in NEED if not os.path.exists(f'artifacts/stage4/{f}')]

assert not missing, (
    f'cell 6 did not finish — {missing} were never written.\n'
    'Nothing is lost and DS2 was NOT opened: the receipt is written last, '
    'so if it is absent the evaluation never completed.\n'
    'Scroll up, read the end of cell 6, and send that text.')

OUT = '/content/stage4_results.zip'
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in NEED:
        z.write(f'artifacts/stage4/{f}', f)

print('packed:', NEED)
files.download(OUT)

packed: ['stage4_results.json', 'RECEIPT.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. What to send back, and what to flag

Send the zip from cell 7 **and the full text output of cell 6**, as text
rather than a screenshot. The output carries the per-seed lines; the summary
table alone loses them.

### Read these before you send

| What | Why it matters |
|---|---|
| `macro-F1` per configuration | Expect roughly **0.50–0.60**. Stage 2's DS1 cross-validated figure was 0.574, and DS2 is a harder, unseen set. |
| `Class S Se / +P` | The whole paper is about this class. Stage 2 reached F1 0.407 on DS1. |
| `Class F` | Will be poor, and that is known in advance — one patient holds 90% of DS1's F beats. Not a surprise and not a bug. |
| The confusion matrix | Look at where S beats go. If they are mostly landing in N, that is the expected failure mode and worth saying so. |

### Stop and report, do not re-run

| Sign | What it means |
|---|---|
| any check failing in cell 4 | the dataset is wrong; the evaluation would be meaningless |
| `macro-F1` above 0.85 | almost certainly a leak — nothing in this project reaches that honestly |
| `macro-F1` below 0.35 | something is wrong with the fit, not with the data |
| the script refusing to run | DS2 has already been opened. **Do not pass `--reason` to get around it.** Tell Rishabh; the receipt will say when and why |
| `AssertionError` of any kind | a guard fired. Send the whole message |

A crash before cell 6 finishes costs nothing — fix it and run again. Once
cell 6 completes, DS2 is open and there is no second first look.